# Notebook 05 — PawPrep Demo Walkthrough

Sets up the environment, verifies the system, and launches the PawPrep Gradio interface.

1. Pull latest code and run Colab setup
2. Verify GPU — download Oxford dataset if not already present
3. Smoke test: generate one illustration before launching the UI
4. Launch the public Gradio URL

> **Run on Colab (T4/V100/A100).** Each generation run takes ~30–40s on T4.

## 0 — Colab Bootstrap

In [ ]:
!git -C /content/stable-diffusion pull
%run /content/stable-diffusion/scripts/colab_setup.py

## 1 — Download Oxford Dataset (if not already present)

Needed for ControlNet Canny conditioning — the app extracts breed-specific edge maps from real Oxford reference photos.
Skip if you already ran `scripts/download_data.py` in a previous session.

In [ ]:
from pathlib import Path

oxford_dir = Path('data/oxford-iiit-pet')
if oxford_dir.exists():
    print(f'Dataset already present at {oxford_dir} ✓')
else:
    print('Downloading Oxford-IIIT Pet dataset (~800 MB)…')
    !python scripts/download_data.py
    print('Download complete ✓')

## 2 — Verify GPU

In [ ]:
import torch

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif device == 'cpu':
    print('⚠️  WARNING: CPU only — generation will be very slow (~10 min per image).')
    print('   Switch to a GPU runtime in Colab: Runtime → Change runtime type → T4 GPU.')

## 3 — Pre-flight Smoke Test

Generate **one** image (fastest possible path) before launching the full UI.  
If this fails, debug here rather than during a live recording.

In [ ]:
from src.app.gradio_app import generate
import matplotlib.pyplot as plt

print('Running smoke test (1 variant, seed=42, beagle / cone_collar / clinic)…')

imgs, pos, neg, ctrl, status = generate(
    animal_type='dog',
    breed='beagle',
    condition='cone_collar',
    environment='clinic',
    style='veterinary illustration',
    use_controlnet=True,
    seed=42,
    uploaded_image=None,
    n_variants=1,   # fast — only 1 image
)

print(f'Status  : {status}')
print(f'Prompt  : {pos[:100]}…')

fig, axes = plt.subplots(1, 2 if ctrl else 1, figsize=(10, 4))
if ctrl:
    axes[0].imshow(ctrl); axes[0].set_title('Control image'); axes[0].axis('off')
    axes[1].imshow(imgs[0]); axes[1].set_title('Generated'); axes[1].axis('off')
else:
    axes.imshow(imgs[0]); axes.set_title('Generated (no ControlNet)'); axes.axis('off')
plt.tight_layout()
plt.show()
print('Smoke test passed ✓')

## 4 — Launch the Gradio App

The cell below starts the server and prints a **public `gradio.live` URL**.  
Open that URL in your browser to use the demo.  
**Copy the URL now** — you'll need it for the video recording.

In [ ]:
from src.app.gradio_app import build_interface

demo = build_interface(data_root='data')

demo.launch(
    share=True,          # generates a public gradio.live URL
    show_error=True,
    quiet=False,
)

# ⬆️  The public URL is printed above. Open it to begin recording.

---

## 5 — Demo Script

Use this when walking through the app live or recording a walkthrough video.

---

### 🎬 Walkthrough (~2–3 minutes)

**[Intro — 15 sec]**
*Show the app: title, disclaimer banner, "How PawPrep helps" bar*
> "PawPrep generates breed-specific veterinary illustrations. Instead of showing a pet owner a generic image when explaining a procedure, the vet can show them what their exact breed will look like."

---

**[Scenario 1 — Pre-surgery briefing — 40 sec]**
*Set: Dog | Beagle | bandaged paw | clinic | veterinary illustration | breed reference ON | seed=42*
*Click Generate*
> "A beagle owner is coming in for a paw injury. Before the visit, the clinic sends them this illustration — they know what to expect. Notice the tri-color coat, floppy ears — it's recognizably a Beagle, not a generic dog."
*Open 'Reference image used' — show the edge map*
> "The shape comes from a real Beagle reference photo. Canny edges guide the pose and proportions."

---

**[Scenario 2 — Post-op cone, personalized — 40 sec]**
*Set: Dog | Great Pyrenees | cone collar | home | veterinary illustration | breed reference ON | seed=100*
*Click Generate*
> "Different breed, different condition. A Great Pyrenees owner is preparing for post-surgery home care. They see their fluffy white dog in the cone — now they know it won't look as dramatic as they imagined."

---

**[Scenario 3 — Upload your own photo — 40 sec]**
*Open upload accordion | Upload a pet photo | keep settings | click Generate*
> "Pet owners can also upload their own photo. PawPrep extracts the pet's silhouette and uses it as the shape guide — so the illustration matches their specific pet's pose, not just the breed average."

---

**[Scenario 4 — Style variation — 30 sec]**
*Set: Cat | Persian | dental check | exam room | educational diagram | seed=137*
*Click Generate*
> "Different styles for different contexts — an educational diagram style works well for clinic handouts or school materials."

---

**[Close — 15 sec]**
> "37 breeds, 8 veterinary conditions, 6 settings. Every combination generates 4 seed variations. All images carry the ethics disclaimer — these are educational illustrations, not clinical guidance."

---

### 🎯 Tips for a clean recording
- Run the smoke test (§3) first so models are cached and generations are faster
- Zoom browser to 90% so the full UI fits without scrolling
- Start recording after the Gradio URL is open and the page has loaded

## 6 — After Recording: Shutdown

In [ ]:
# Run this cell to gracefully shut down the Gradio server after recording
demo.close()
print('Gradio server stopped.')

---
**End of Notebook 05.**